# Jena Weather Forecasting - Tutorial 2 Demo

**EMI-3** | António Cruz, Bruno Santos, Pedro Miranda, Ricardo Kayseller

This notebook demonstrates the complete ML pipeline for temperature forecasting using the Jena Climate dataset. It calls the modular scripts in `src/` and tracks everything with MLflow via noted's auto-instrumentation.

**Pipeline stages:** Ingestion -> Preprocessing -> Training -> Evaluation

## 1. Setup

Import modules and load the Hydra configuration.
The `cfg` object is auto-injected by noted's config selector.

In [1]:
import sys, os
import numpy as np
import pandas as pd
import mlflow

# Project root for imports
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ingestion.ingest import ingest
from src.preprocessing.preprocess import preprocess
from src.training.train import train
from src.evaluation.evaluate import evaluate

# cfg is injected by noted's Hydra config selector
print(f"Model type: {cfg.model.type}")
print(f"Epochs: {cfg.training.epochs}")
print(f"Batch size: {cfg.training.batch_size}")
print(f"Learning rate: {cfg.training.learning_rate}")

## 2. Data Ingestion

Load the Jena Climate CSV (420k+ records, 2009-2016, 10-min intervals).

In [2]:
df_raw = ingest(cfg, project_root=project_root)

print(f"\nShape: {df_raw.shape}")
print(f"Features: {list(df_raw.columns)}")
df_raw.head()

## 3. Preprocessing

Resample to hourly, add cyclical time features, standardize,
and create sliding-window sequences for the model.

In [3]:
prep = preprocess(df_raw, cfg)

print(f"\nInput shape:  X_train={prep['X_train'].shape}")
print(f"Output shape: y_train={prep['y_train'].shape}")
print(f"Features: {prep['feature_cols']}")
print(f"Target: {prep['target_col']}")
print(f"Sequence length: {prep['X_train'].shape[1]} hours")
print(f"Forecast horizon: {prep['y_train'].shape[1]} hours")

## 4. Training

Train the model with MLflow tracking.
Watch the **Live Metrics** panel for real-time loss curves.

In [4]:
result = train(prep, cfg)

print(f"\nFinal test metrics:")
for k, v in result['test_metrics'].items():
    print(f"  {k}: {v:.4f}")

## 5. Evaluation

Generate prediction plots and log artifacts to MLflow.

In [5]:
eval_result = evaluate(result, cfg)

print(f"\nModel registered: {eval_result['registered']}")
if eval_result['model_version']:
    print(f"Registry version: {eval_result['model_version']}")

## 6. Prediction Scatter Plot

Scatter plot of predicted vs actual temperature to assess overall model accuracy.

In [6]:
import matplotlib.pyplot as plt
import numpy as np

y_true = result['y_test_c'].reshape(-1)
y_pred = result['y_pred_c'].reshape(-1)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.15, s=4, color='#4a90d9')

# Ideal prediction line
lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
ax.plot(lims, lims, '--', color='#c0392b', linewidth=1.5, label='Ideal')

ax.set_xlabel('Actual Temperature (degC)')
ax.set_ylabel('Predicted Temperature (degC)')
ax.set_title(f'{cfg.model.type} - Predicted vs Actual')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()